# derived_8.4-hybrid-lstm-1.4 — V23 LSTM + PCA + XGBoost with Accelerated SHAP

This experiment replaces the V9-style LSTM with **V23 BiLSTM+Attn** (top25 curated features, seq_len=30, ReduceLROnPlateau, 5-seed training). After training, PCA is applied to all three frozen representations (160-dim ctx, 80-dim head_hidden, 80-dim pre-ReLU) at three compression levels (95% variance, 64 components, 32 components) to evaluate how dimensionality reduction affects hybrid model performance.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

NOTEBOOK_DIR = Path.cwd() / "experiment/derived_8.4-hybrid-lstm-1.4"
PROJECT_ROOT = NOTEBOOK_DIR.parents[2]
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(NOTEBOOK_DIR))

from lstm.train import (
    train_v23_and_extract_all, TOP25_FEATURES, SEQ_LEN,
)
from eval_hybrid.data import load_hybrid_experiment_data
from eval_hybrid.evaluator import HybridStrategyEvaluator
from eval_hybrid.shap_analysis import run_full_shap_analysis

with open(NOTEBOOK_DIR / "config.yaml") as f:
    config = yaml.safe_load(f)

ARTIFACTS_DIR = NOTEBOOK_DIR / "artifacts"
MODELS_DIR = NOTEBOOK_DIR / "models"
DATA_DIR = PROJECT_ROOT / config.get("data_dir", "data/splits/derived_8.4")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete.")

## Phase 1-3: V23 Training + Extraction + PCA

Trains V23 BiLSTM+Attn (5 seeds, best by val RMSE), extracts raw ctx/hh/hp vectors, computes PCA at 3 levels.

In [ ]:
import json

has_raw = all((ARTIFACTS_DIR / f"{n}_{s}.npy").exists()
              for n in ("ctx", "head_hidden", "head_pre_relu")
              for s in ("train", "val", "test"))

from lstm.train import PCA_LEVELS as _PLEVELS
_pca_raw = ("ctx", "head_hidden", "head_pre_relu")
_pca_lvls = [lvl for lvl, _ in _PLEVELS]
has_pca = all((ARTIFACTS_DIR / f"{n}_{l}_{s}.npy").exists()
              for n in _pca_raw
              for l in _pca_lvls
              for s in ("train", "val", "test"))

if has_raw and has_pca:
    print("Found existing representations. Skipping V23 training.")
    with open(ARTIFACTS_DIR / "lstm_metrics.json") as f:
        lstm_metrics = json.load(f)
else:
    best_seed_info, pca_info, lstm_metrics = train_v23_and_extract_all(DATA_DIR, ARTIFACTS_DIR, MODELS_DIR)
    if best_seed_info:
        print(f"Best seed: {best_seed_info['seed']} (val_rmse={best_seed_info['val_rmse']:.5f})")
print(f"LSTM-only test R2 = {lstm_metrics['test']['r2']:.4f}")

## Phase 4-5: XGBoost Tabular Baselines + Hybrid Models

Train 2 tabular baselines and 24 hybrid models (12 representation variants x 2 strategies).

In [ ]:
from eval_hybrid.evaluator import HybridStrategyEvaluator
from run_eval import compute_c1_gain_additions, VARIANTS, _model_name, _candidate_id, _repr_suffix, REPR_LABELS, PCA_DISPLAY

c1_additions = compute_c1_gain_additions(config)
backbone_54 = config["shared_backbone_54"]

data_base = load_hybrid_experiment_data(PROJECT_ROOT, NOTEBOOK_DIR, config, repr_type="ctx", pca_suffix="")
eval_global = HybridStrategyEvaluator(data_base, config, "Global_Single", models_dir=MODELS_DIR)
eval_v0 = HybridStrategyEvaluator(data_base, config, "Clustering_V0_Full_k2", models_dir=MODELS_DIR)

results = {}
results["Global Single (54 Backbone)"] = eval_global.fit_and_evaluate(
    model_name="Global Single (54 Backbone)", candidate_id="Global_Single_54_Backbone",
    global_features=backbone_54,
)
results["Clustering_V0_Full_k2 (54 Backbone, c0=0, c1=10)"] = eval_v0.fit_and_evaluate(
    model_name="Clustering_V0_Full_k2 (54 Backbone, c0=0, c1=10)",
    candidate_id="Clustering_V0_k2_54_Backbone", global_features=backbone_54,
    cluster_additions={"0": [], "1": c1_additions},
)

for repr_type, pca_level in VARIANTS:
    suffix = _repr_suffix(pca_level)
    data_v = load_hybrid_experiment_data(PROJECT_ROOT, NOTEBOOK_DIR, config, repr_type=repr_type, pca_suffix=suffix)
    ev_g = HybridStrategyEvaluator(data_v, config, "Global_Single", models_dir=MODELS_DIR)
    ev_c = HybridStrategyEvaluator(data_v, config, "Clustering_V0_Full_k2", models_dir=MODELS_DIR)
    results[_model_name(repr_type, pca_level, "Global_Single")] = ev_g.fit_and_evaluate(
        model_name=_model_name(repr_type, pca_level, "Global_Single"),
        candidate_id=_candidate_id(repr_type, pca_level, "Global_Single"),
        global_features=data_v.hybrid_features,
    )
    results[_model_name(repr_type, pca_level, "Clustering_V0_Full_k2")] = ev_c.fit_and_evaluate(
        model_name=_model_name(repr_type, pca_level, "Clustering_V0_Full_k2"),
        candidate_id=_candidate_id(repr_type, pca_level, "Clustering_V0_Full_k2"),
        global_features=data_v.hybrid_features,
        cluster_additions={"0": [], "1": c1_additions},
    )

print(f"Trained {len(results)} XGBoost models.")

## Phase 6: SHAP Feature Importance Analysis

Accelerated SHAP using XGBoost native pred_contribs (C++/CUDA tree traversal).

In [ ]:
# Build per-model evaluator map for SHAP
eval_map = {}
for rtype, plevel in VARIANTS:
    suffix = _repr_suffix(plevel)
    dv = load_hybrid_experiment_data(PROJECT_ROOT, NOTEBOOK_DIR, config, repr_type=rtype, pca_suffix=suffix)
    ev_g = HybridStrategyEvaluator(dv, config, "Global_Single", models_dir=MODELS_DIR)
    ev_c = HybridStrategyEvaluator(dv, config, "Clustering_V0_Full_k2", models_dir=MODELS_DIR)
    eval_map[_model_name(rtype, plevel, "Global_Single")] = ev_g
    eval_map[_model_name(rtype, plevel, "Clustering_V0_Full_k2")] = ev_c

eval_map["Global Single (54 Backbone)"] = eval_global
eval_map["Clustering_V0_Full_k2 (54 Backbone, c0=0, c1=10)"] = eval_v0

shap_info = run_full_shap_analysis(eval_map, results, ARTIFACTS_DIR)
print(f"SHAP computed for {len(shap_info['shap_results'])} models.")

## Results: Leaderboard

In [ ]:
records = [r.as_record() for r in results.values()]
records.append({
    "model_name": "BiLSTM+Attn (LSTM-only, V23)",
    "pooled_r2": lstm_metrics["test"]["r2"],
    "pooled_rmse": lstm_metrics["test"]["rmse"],
    "pooled_ubrmse": lstm_metrics["test"]["ubrmse"],
    "pooled_bias": lstm_metrics["test"]["bias"],
    "pooled_mae": lstm_metrics["test"]["mae"],
    "pooled_pearson": float("nan"),
})
df = pd.DataFrame(records).sort_values("pooled_r2", ascending=False)
display_cols = ["model_name", "pooled_r2", "pooled_rmse", "pooled_mae", "pooled_pearson"]
display(df[display_cols].round(4))

## Save Artifacts & Generate README

In [ ]:
df.to_csv(ARTIFACTS_DIR / "summary_records.csv", index=False)
with open(ARTIFACTS_DIR / "metrics.json", "w") as f:
    json.dump({k: r.as_record() for k, r in results.items()}, f, indent=2)

from run_eval import generate_readme
df_regime = pd.DataFrame()
generate_readme(df, df_regime, shap_info)
print("README generated.")